In [1]:
import pandas as pd

r_cols = ['user_id', 'movie_id', 'rating'] # Nomes das colunas

ratings = pd.read_csv('codigos/u.data', sep='\t', names=r_cols, usecols=range(3)) # leitura do csv u.data dentro da pasta codigos, aplicando as colunas que queremos ler

ratings.head() # mostrando o topo do dataframe

,user_id,movie_id,rating
0,0,50,5
1,0,172,5
2,0,133,1
3,196,242,3
4,186,302,3


In [2]:
import numpy as np
# agrupando pelo id do filmem e agregando as avalicaoes com a quantidade\ de vezes que o filme foi avaliado e a media das avaliacoes
movieProperties = ratings.groupby('movie_id').agg({'rating': [np.size, np.mean]}) # numero de avaliações e medias de cada filmes
movieProperties.head()

C:\Users\User\AppData\Local\Temp\ipykernel_12508\3644794803.py:3: FutureWarning: The provided callable <function mean at 0x000001CB979A4680> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  movieProperties = ratings.groupby('movie_id').agg({'rating': [np.size, np.mean]}) # numero de avaliações e medias de cada filmes


rating          
           size      mean
movie_id                 
1           452  3.878319
2           131  3.206107
3            90  3.033333
4           209  3.550239
5            86  3.302326

In [4]:
# Criando dataframe com numero de avaliações dos filmes e quantidade de vezes que foi avaliado
movieNumRatings = pd.DataFrame(movieProperties['rating']['size'])
# normalizando o número de avaliacoes onde o valor mínimo será 0 e o máximo será 1
movieNormalizedNumRatings = movieNumRatings.apply(lambda x: (x - np.min(x)) / (np.max(x) - np.min(x))) 
movieNormalizedNumRatings.head()

,size
movie_id,
1,0.773585
2,0.222985
3,0.152659
4,0.356775
5,0.145798


In [15]:
movieDict = {} # criando um dicionário com as informações dos 

# abrindo o arquivo com as info dos filmes, com encoding latin-1 pq tava dando erro com caracteres especiais no utf-8
with open(r'codigos/u.item', encoding='latin-1') as f:
    temp = ''
    for line in f:  # para cada linha do arquivo
        # campos separados por |
        fields = line.rstrip('\n').split('|')
        #pegando as informações do filme
        movieID = int(fields[0])
        name = fields[1]
        genres = fields[5:25]
         # convertendo os generos para uma lista de int
        genres = list(map(int, genres))
        # adicionando as informações no dicionário
        movieDict[movieID] = (name, genres, movieNormalizedNumRatings.loc[movieID].get('size'), movieProperties.loc[movieID].rating.get('mean'))
    

In [16]:
movieDict[1]

('Toy Story (1995)',
 [0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 np.float64(0.7735849056603774),
 np.float64(3.8783185840707963))

In [17]:
from scipy import spatial

def ComputeDistance(a, b):
    
    genresA = a[1] # genero A
    genresB = b[1] # genero B

    #Comparando a distancia do genero utilizando o cosseno para dessemelhança 
    genreDistance = spatial.distance.cosine(genresA, genresB)
    
    popularityA = a[2] # popularidade A
    popularityB = b[2] # popularidade B
    popularityDistance = abs(popularityA - popularityB)
    return genreDistance + popularityDistance # retorna a distancia do genero + a distancia da popularidade

# testando com dois filmes diferentes
ComputeDistance(movieDict[2], movieDict[4])

np.float64(0.8004574042309892)

In [19]:
print(movieDict[2])
print(movieDict[4])

('GoldenEye (1995)', [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], np.float64(0.22298456260720412), np.float64(3.2061068702290076))
('Get Shorty (1995)', [0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], np.float64(0.3567753001715266), np.float64(3.550239234449761))


In [29]:
import operator

# Pega nos filmes "vizinhos" do filme passado como parametro
def getNeighbors(movieID,K):
    distances = []    # uma lista de tuplas com as distâncias e o filme
    
    for movie in movieDict: # para cada filme no dicionário
       # se o filme for diferente do filme passado como parametro
        if (movie != movieID):
            dist = ComputeDistance(movieDict[movieID], movieDict[movie]) # calcula a distancia entre os dois filmes
            # adiciona na lista de distancias o filme e a distancia
            distances.append((movie, dist))
    distances.sort(key=operator.itemgetter(1))  # ordena a lista de distancias pelo segundo item da tupla (a distancia
    
    neighbors = [] # pega os K vizinhos mais próximos
    for x in range(K):
        neighbors.append(distances[x][0])
    return neighbors

# teste prático com o filme 1
K = 10
avgRating = 0
neighbors = getNeighbors(1,K)
for neighbor in neighbors:
    avgRating += movieDict[neighbor][3]
    print(movieDict[neighbor][0] + " " + str(movieDict[neighbor][3]))

avgRating /= float(K)

Liar Liar (1997) 3.156701030927835
Aladdin (1992) 3.8127853881278537
Willy Wonka and the Chocolate Factory (1971) 3.6319018404907975
Monty Python and the Holy Grail (1974) 4.0664556962025316
Full Monty, The (1997) 3.926984126984127
George of the Jungle (1997) 2.685185185185185
Beavis and Butt-head Do America (1996) 2.7884615384615383
Birdcage, The (1996) 3.4436860068259385
Home Alone (1990) 3.0875912408759123
Aladdin and the King of Thieves (1996) 2.8461538461538463


In [30]:
avgRating

np.float64(3.3445905900235564)

In [31]:
movieDict[1]

('Toy Story (1995)',
 [0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 np.float64(0.7735849056603774),
 np.float64(3.8783185840707963))

## Activity
1. Our choice of 10 for K was somewhat arbitrary - what effect do different K values have on the results?

2. Our distance metric was also somewhat arbitrary - we just took the cosine distance between the genres and added it to the difference between the normalized popularity scores. Can you improve on tha

In [32]:
# K é apenas a quantidade de vizinhos que serão retornados da função getNeighbors. 
# Isso impacta na quantidade que vai ser mostrado depois mas também na média final das avaliações (rating).

In [33]:
from scipy import spatial

def ComputeDistance(a, b):
    
    genresA = a[1] # genero A
    genresB = b[1] # genero B

    #Comparando a distancia do genero utilizando o cosseno para dessemelhança 
    genreDistance = spatial.distance.cosine(genresA, genresB)
    
    popularityA = a[2] # popularidade A
    popularityB = b[2] # popularidade B
    popularityDistance = abs(popularityA - popularityB) # distancia da popularidade

    # avaliação dos filmes a e b
    ratingA = a[3]
    ratingB = b[3]
    # distantica das avaliações entre os filmes
    # divindo por 5(pardronizado de avaliações de até 5 estrelas) para normalizar o valor entre 0 e 1
    ratingDistance = abs(ratingA - ratingB) / 5.

    # colocando pesos diferentes para cada atributo
    genre_weight = 0.6
    popularity_weight = 0.2
    rating_weight = 0.2
    
     # retorna a distancia do genero + a distancia da popularidade + avaliação
    return (genreDistance * genre_weight) + (popularityDistance * popularity_weight) + (ratingDistance * rating_weight)

ComputeDistance(movieDict[2], movieDict[4])

np.float64(0.44052344208169464)